In [1]:
!nvidia-smi

Wed May 13 03:17:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%pip install -q fastapi uvicorn pyngrok python-dotenv transformers accelerate sentencepiece huggingface_hub torch requests
%pip install -U -q "bitsandbytes>=0.46.1"

In [6]:
%cd /content
!git clone https://github.com/Nguyenphamtanan/medical-chat-demo
%cd medical-chat-demo/ai-service

/content
Cloning into 'medical-chat-demo'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (127/127), done.
remote: Total 175 (delta 57), reused 159 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (175/175), 110.89 KiB | 4.26 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/medical-chat-demo/ai-service


In [6]:
%pip install -q fastapi uvicorn pyngrok python-dotenv transformers accelerate sentencepiece huggingface_hub torch

In [10]:
%cd /content/medical-chat-demo
!git reset --hard
!git pull
!git log --oneline -3
%cd ai-service

/content/medical-chat-demo
HEAD is now at 0740d89 add full agent advanced case analysis mode
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 564 bytes | 564.00 KiB/s, done.
From https://github.com/Nguyenphamtanan/medical-chat-demo
   0740d89..785bb76  main       -> origin/main
Updating 0740d89..785bb76
Fast-forward
 ai-service/app/advanced_agent/knowledge_base.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)
785bb76 (HEAD -> main, origin/main, origin/HEAD)  fix unicode token regex in advanced agent
0740d89 add full agent advanced case analysis mode
164df34 speed up medgemma web flow and avoid repair timeout
/content/medical-chat-demo/ai-service


In [5]:
%cd medical-chat-demo/ai-service


[Errno 2] No such file or directory: 'medical-chat-demo/ai-service'
/content


In [6]:
!pwd
!ls
!ls app/services

/content/medical-chat-demo/ai-service
app  notebooks	requirements.txt
medgemma_service.py  orchestrator.py  parser_service.py  prompt_service.py


In [11]:
!grep -n "def repair_json" app/services/medgemma_service.py
!grep -n "build_json_repair_prompt" app/services/prompt_service.py
!grep -n "build_non_json_medgemma_response" app/services/parser_service.py

193:    def repair_json(self, prompt: str) -> Dict[str, str]:
39:def build_json_repair_prompt(symptoms: str, raw_output: str) -> str:
286:def build_non_json_medgemma_response(symptoms: str, raw_text: str) -> Dict[str, Any]:


In [2]:
%pip install -q fastapi uvicorn pyngrok python-dotenv transformers accelerate sentencepiece huggingface_hub torch requests

In [8]:
%pip install -r requirements.txt

In [3]:
import os
from huggingface_hub import login, whoami

hf_token = input("Paste HF_TOKEN here: ").strip()

os.environ["HF_TOKEN"] = hf_token
os.environ["USE_MEDGEMMA"] = "true"
os.environ["MEDGEMMA_MODEL_ID"] = "google/medgemma-1.5-4b-it"

login(token=hf_token, add_to_git_credential=False)

print("HF login done")
print(whoami())

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login done
{'type': 'user', 'id': '68b94413adff262556a73fb6', 'name': 'tran0398', 'fullname': 'Loc Hoang Tran', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1780272000, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/rGLdqgbj5NlqdNv87lg2V.png', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'Loc Hoang Tran', 'role': 'fineGrained', 'createdAt': '2026-04-20T02:29:49.189Z', 'fineGrained': {'canReadGatedRepos': True, 'global': ['discussion.write', 'post.write'], 'scoped': [{'entity': {'_id': '68b94413adff262556a73fb6', 'type': 'user', 'name': 'tran0398'}, 'permissions': ['repo.content.read', 'repo.access.read', 'repo.write', 'inference.serverless.write', 'inference.endpoints.infer.write', 'inference.endpoints.write', 'user.webhooks.read', 'user.webhooks.write', 'collection.read', 'collection.write', 'discussion.write', 'user.billing.read', 'job.write']}]}}}}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [6]:
!python -c "from app.main import app; print('FastAPI app OK')"

FastAPI app OK


In [29]:
!lsof -ti:8000 | xargs -r kill -9

: 

: 

: 

In [ ]:
import threading
import uvicorn

def run_api():
    uvicorn.run("app.main:app", host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_api, daemon=True)
thread.start()

print("FastAPI started on port 8000")

FastAPI started on port 8000


INFO:     Started server process [1400]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [5]:
%cd /content/medical-chat-demo/ai-service
!python -c "from app.main import app; print('FastAPI app OK')"

[Errno 2] No such file or directory: '/content/medical-chat-demo/ai-service'
/content
Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'app'


In [6]:
!lsof -ti:8000 | xargs -r kill -9

In [8]:
from pyngrok import ngrok

ngrok_token = input("Paste NGROK_AUTH_TOKEN here: ").strip()
ngrok.set_auth_token(ngrok_token)

# Dọn tunnel cũ nếu có
ngrok.kill()

public_url = ngrok.connect(8000)
print("PUBLIC_URL:", public_url)
print("COPY_THIS:", public_url.public_url)

PUBLIC_URL: NgrokTunnel: "https://02a5-34-73-217-207.ngrok-free.app" -> "http://localhost:8000"     
COPY_THIS: https://02a5-34-73-217-207.ngrok-free.app


In [1]:
!fuser -k 8000/tcp || true

8000/tcp:             6610


In [12]:
import requests

res = requests.get("http://127.0.0.1:8000/docs", timeout=10)
print(res.status_code)

INFO:     127.0.0.1:58272 - "GET /docs HTTP/1.1" 200 OK
200


In [ ]:
import requests

base_url = "https://02a5-34-73-217-207.ngrok-free.app"

res = requests.post(
    f"{base_url}/ai/analyze",
    json={"symptoms": "Tôi bị sốt, ho, đau họng 2 ngày nay"},
    timeout=600
)

print(res.status_code)
print(res.text[:3000])

404
<!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Medium-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-MediumItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/ibm-plex-mono/IBMPlexMono-Text.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="prelo

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[MEDGEMMA] CUDA available: Tesla T4
[MEDGEMMA] Loading model: google/medgemma-1.5-4b-it


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


...

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=384) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[MEDGEMMA] Model loaded successfully.
[MEDGEMMA] Calling real model...


In [3]:
%cd /content/medical-chat-demo
!git pull
%cd ai-service

/content/medical-chat-demo
Updating 25e413b..78ebf71
error: Your local changes to the following files would be overwritten by merge:
	ai-service/app/services/orchestrator.py
Please commit your changes or stash them before you merge.
Aborting
/content/medical-chat-demo/ai-service


In [15]:
import os

print("USE_MEDGEMMA =", os.getenv("USE_MEDGEMMA"))
print("HF_TOKEN length =", len(os.getenv("HF_TOKEN", "")))
print("MODEL =", os.getenv("MEDGEMMA_MODEL_ID"))

USE_MEDGEMMA = true
HF_TOKEN length = 37
MODEL = google/medgemma-1.5-4b-it


In [11]:
import requests

text = """
Tôi là nữ, 52 tuổi. Khoảng 1 tuần nay tôi thấy da và mắt hơi vàng, người mệt nhiều, ăn kém, ngứa toàn thân và nước tiểu sẫm màu hơn bình thường. Tôi không đau ngực, không khó thở, nhưng đôi lúc hơi buồn nôn và tức nhẹ vùng bụng trên bên phải. Tôi có tiền sử gan nhiễm mỡ và đái tháo đường type 2, đang dùng metformin. Gần đây tôi có uống thêm thực phẩm chức năng giảm cân. Tôi muốn biết tình trạng này có thể liên quan đến cơ quan nào, có dấu hiệu nguy hiểm nào cần đi khám sớm không?
""".strip()

res = requests.post(
    "http://127.0.0.1:8000/ai/analyze",
    json={"symptoms": text},
    timeout=600
)

print(res.status_code)
print(res.text[:5000])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


[MEDGEMMA] CUDA available: Tesla T4
[MEDGEMMA] Loading model: google/medgemma-1.5-4b-it


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=384) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[MEDGEMMA] Model loaded successfully.
[MEDGEMMA] Calling real model...
========== RAW MEDGEMMA OUTPUT ==========
<unused94>thought
The user is a 52-year-old female presenting with symptoms including:
- Yellowish skin and eyes (jaundice)
- Fatigue
- Poor appetite
- Generalized itching
- Darker urine
- Occasional nausea and mild upper right abdominal discomfort
- History of fatty liver disease and type 2 diabetes (on metformin)
- Recent use of a weight loss supplement

The user wants to know which organ system might be involved and if there are any dangerous signs requiring urgent attention.

I need to create a JSON object with the specified keys:
- `summary`: A concise summary of the user's symptoms.
- `possible_related_systems`: Systems potentially involved based on symptoms.
- `possible_explanations`: Potential causes/explanations for the symptoms (without diagnosis).
- `red_flags`: Specific signs indicating a need for urgent medical attention.
- `missing_questions`: Questions to ask 